# Langfuse 02 · 评估与打分（Score / RAG 评估 / Agent 评估）

追踪（01 课）回答「发生了什么」，评估回答「**好不好**」。两者合起来才是评估闭环：

| 环节 | 回答的问题 | 产物 |
|---|---|---|
| trace（链路） | 每一步的输入输出是什么 | 一条可点进去看的调用链 |
| score（分数） | 这次调用好还是坏 | 一个可按 name 聚合、可筛选的字段 |

本 notebook 把「监控与评估 → 评估」这一节的三课全部串起来，一次跑完三种评估：

| 小节 | 评估对象 | 判分方式 |
|---|---|---|
| 1 · 打分 Score | 一次调用（回复长度、用户点赞/点踩、数据集批量） | 数值 / 类别型分数 |
| 2 · RAG 评估 | 检索质量 + 生成质量（四项指标） | LLM-as-Judge |
| 3 · Agent 评估 | 行为（主题、工具调用、目标达成） | 精确匹配 / 混淆矩阵 / LLM 裁判 |

> **本 notebook 由 `Agent/06_langfuse/` 下 5 个脚本合并而成**：
> `04_评估_打分.py`（课案原版）、`04_评估_打分_jxsd.py`（完整版）、
> `05_评估_RAG与Agent.py`（原版）、`05_评估_RAG与Agent_jxsd.py`（完整版）、
> `06_评估_Agent指标_jxsd.py`（新增完整版，789 行，全仓最长）。

**官方文档**
- 评估总览：<https://langfuse.com/docs/evaluation/overview>
- 打分（Scores）：<https://langfuse.com/docs/evaluation/features/scores>
- 实验（Experiments / RAG 评估）：<https://langfuse.com/docs/evaluation/experiments/experiments>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型（裁判 + 被测 Agent） |
| 依赖 | `langfuse`（SDK v4.15.1）/ `deepagents` / `langchain` / `langchain-openai`（venv 已装） |
| 密钥 | `settings.api_key`（已配置）；`settings.langfuse_public_key` / `langfuse_secret_key`（已配置，可空） |
| 前置服务 | 无（Langfuse **可空**：密钥为空时全篇走「打印报文」降级路径，分数照常算） |
| 预计耗时 | 约 4~9 分钟（**全仓最慢的一课**，绝大部分时间在逐条真调大模型） |

> ⚠️ 本机 Langfuse 是 **v4 的 events_only 模式**（`http://localhost:3001`）。
> 分数**上传**（`create_score` / `score_current_trace`）走事件通道、照常可用；
> 分数**取回**（`GET /api/public/v2/scores`）在 v4 events_only 下不存在，程序取数要走 metrics 接口。
> 本课只「打分 + 上传」，不「取回」，所以不受这个限制影响；具体坑见文末「常见坑」。

三个小节各自都带了**降级路径**：Langfuse 密钥为空、或 `ragas` 没装，都不会崩 ——
分数照常真算真打印，只是「上报」这一步换成打印「本应发送的报文」。

## 本节地图

```mermaid
graph TD
    A["1. 打分 Score<br/>把好坏变成可筛选字段"] --> D["上传 Langfuse<br/>create_score / score_current_trace"]
    B["2. RAG 评估<br/>检索 CP/CR + 生成 Faith/AR"] --> D
    C["3. Agent 评估<br/>主题 / 工具调用 / 目标达成"] --> D
    D --> E["看板按 name 聚合<br/>哪个版本掉了、哪类问题最多"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 小节 | 评估对象 | 关键指标 | 值域 |
|---|---|---|---|
| 1 · 打分 | 一次调用 | response_length_ok / user_feedback / accuracy | 0~1 或类别 |
| 2 · RAG 评估 | 检索 + 生成 | Context Precision / Recall、Faithfulness、Answer Relevancy | 0~1 |
| 3 · Agent 评估 | 行为轨迹 | Topic Adherence / Tool Call Accuracy / Tool Call F1 / Agent Goal Accuracy | 0~1 或 0/1 |

**上下游衔接**：本课是「自动打分」三连（04/05/06）——
下一课 `03_标注与数据.ipynb` 会拿本课「自动打分偏低」的那批 trace 去**人工标注**，
沉淀成数据集后再回到本课做回归。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课**不往磁盘写任何文件**（分数、数据集都在 Langfuse 服务端，Agent 也只读内存），
> 所以 `WORKDIR` 只是照抄模板留着的；并发跑同章其它 notebook 不会互相踩文件。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\06_langfuse\tmp_nb_work
```

## 0.1 前置条件自检

本课是 🟡 需模型档。这一格只做一件事：**把「本机到底有什么」先打出来**，
这样后面哪一段走了降级路径，你一眼就知道为什么。

检查五样东西：

| 检查项 | 缺了会怎样 |
|---|---|
| `langfuse` 包 | 无法上传分数（本机 venv 已装 4.15.1） |
| `deepagents` 包 | 第 1、3 节被测 Agent 建不起来 |
| `ragas` 包 | 第 2 节走「本地 LLM-as-Judge 近似版」（装了才用官方指标） |
| Langfuse 密钥 | 为空 → 全篇走「打印报文」降级路径 |
| 模型 `api_key` | 全篇依赖它（裁判 + Agent 都真调） |

> **不打印密钥值**，只看「在不在」。

In [ ]:
# 前置条件自检：只打印现状，不抛异常 —— 缺什么，后面走降级分支时你能立刻对上号。
import socket
import urllib.parse

from config import settings

_todo: list[str] = []

try:
    import langfuse

    print(f"[OK] langfuse {getattr(langfuse, '__version__', '?')} 已装 —— 分数上传可用")
except ImportError:
    _todo.append("langfuse")
    print("[跳过] 缺 langfuse：先 `uv add langfuse`，本 notebook 上传分数需要它")

try:
    import deepagents  # noqa: F401

    print(f"[OK] deepagents {getattr(deepagents, '__version__', '?')} 已装 —— 被测 Agent 可用")
except ImportError:
    _todo.append("deepagents")
    print("[跳过] 缺 deepagents：先 `uv add deepagents`，第 1、3 节要建 Agent")

try:
    from ragas.metrics.collections import AnswerRelevancy  # noqa: F401

    _ragas_ok = True
except ImportError as exc:
    _ragas_ok = False
    print(f"[信息] ragas 未装（{type(exc).__name__}）—— 第 2 节走本地 LLM-as-Judge 近似版")

_pk_ok = bool(settings.langfuse_public_key)
_sk_ok = bool(settings.langfuse_secret_key)
print(f"[信息] LANGFUSE_PUBLIC_KEY = {'已配置' if _pk_ok else '未配置'}")
print(f"[信息] LANGFUSE_SECRET_KEY = {'已配置' if _sk_ok else '未配置'}")
print(f"[信息] LANGFUSE_HOST       = {settings.langfuse_host}")

_parsed = urllib.parse.urlparse(settings.langfuse_host)
_lf_host = _parsed.hostname or "localhost"
_lf_port = _parsed.port or (443 if _parsed.scheme == "https" else 80)
try:
    with socket.create_connection((_lf_host, _lf_port), timeout=2.0):
        print(f"[OK] {_lf_host}:{_lf_port} 已连通 —— 分数会真的写进这台 Langfuse")
except OSError as exc:
    print(f"[跳过] {_lf_host}:{_lf_port} 连不上（{type(exc).__name__}）—— "
          "若密钥已配置，请先把 Langfuse 起起来")

print(f"[信息] 模型 {settings.model_name} 的 api_key = "
      f"{'已配置' if settings.api_key else '未配置'}（三个小节都真调它）")

if not (_pk_ok and _sk_ok):
    print("[跳过] Langfuse 密钥为空：全篇走「打印 score 报文 + 本地桩对象」的降级路径")
if _todo:
    print(f"[跳过] 缺少：{_todo}")

### 预期输出

本机（Langfuse 已配好、3001 端口在听、ragas 未装）会看到：

```text
[OK] langfuse 4.15.1 已装 —— 分数上传可用
[OK] deepagents 0.7.13 已装 —— 被测 Agent 可用
[信息] ragas 未装（ModuleNotFoundError）—— 第 2 节走本地 LLM-as-Judge 近似版
[信息] LANGFUSE_PUBLIC_KEY = 已配置
[信息] LANGFUSE_SECRET_KEY = 已配置
[信息] LANGFUSE_HOST       = http://localhost:3001
[OK] localhost:3001 已连通 —— 分数会真的写进这台 Langfuse
[信息] 模型 deepseek-flash 的 api_key = 已配置（三个小节都真调它）
```

> ⚠️ `deepagents` 与 `ragas` 的版本号、`ModuleNotFoundError` 后面的具体模块名，
> 以你运行时为准；上面只是本机这次的实测值。

## 1. 打分 Score：把「主观好坏」变成可筛选字段

一条 score 的四个要素：

| 字段 | 说明 |
|---|---|
| `trace_id` | 这个分数打在**哪一次调用**上（也可以用 `observation_id` 打到某个 span） |
| `name` | 指标名，例如 `accuracy` / `response_length_ok` / `user_feedback` |
| `value` | 数值或字符串；配合 `data_type` 决定语义 |
| `comment` | 说明 / 理由，排查时最关键的一列 |

分数从哪来？分两类：

| 来源 | 谁打分 | 例子 |
|---|---|---|
| 人工打分 | 人在 Web UI 上点 | 点赞/点踩、给客服回答打 1~5 分（标注队列，见 03 课） |
| 代码打分 | 程序 | 用户反馈上报、业务指标、字符串匹配、LLM 评审 |

### 1.1 课案原版：`score_current_trace` 最短实现

v4 的关键 API 是 `score_current_trace()`：**在 `@observe` 函数里直接调用即可**，
SDK 知道你当前跑在哪个 span 里，不需要手动拿 `trace_id`。

In [ ]:
from langchain.chat_models import init_chat_model
from langfuse import Langfuse, observe
from config import settings

lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    host=settings.langfuse_host,
)

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@observe
def answer(question: str) -> str:
    """带追踪的业务函数：内部收集业务指标并打分"""
    response = llm.invoke(question)

    # ---------- 业务指标打分示例：回复长度是否达标 ----------
    length = len(response.content)
    lf.score_current_trace(
        name="response_length_ok",
        value=1.0 if 20 <= length <= 200 else 0.0,
        comment=f"回复长度 {length} 字",
    )
    return response.content


@observe
def with_user_feedback(question: str) -> str:
    """模拟「用户点了踩」：把点赞/点踩上报为 trace 分数"""
    response = llm.invoke(question)
    feedback = "thumb_up"  # 实际来自前端用户的点赞按钮
    lf.score_current_trace(
        name="user_feedback",
        value=feedback,          # 类别型分数直接传字符串
        data_type="CATEGORICAL",
    )
    return response.content

In [ ]:
answer("什么是 Function Call？一句话回答")
with_user_feedback("什么是 MCP？一句话回答")
lf.flush()
print("打分已上传，在 Langfuse 的 trace 详情和看板中查看")

### 预期输出

```text
打分已上传，在 Langfuse 的 trace 详情和看板中查看
```

> ⚠️ 两条回答的正文由模型决定、每次不同；这里唯一确定的是最后一行「打分已上传」，
> 以及「去 Langfuse trace 详情页能看到 `response_length_ok` / `user_feedback` 两条分」这件事。

### 1.2 完整版：数据集自动打分 + 在 `@observe` 里上报业务指标

原版只演示了「在函数里打分」。完整版补上两块：

1. **数据集自动打分**：先建数据集（`input` + `expected_output`），逐条跑 Agent，
   用「期望输出是不是回答的子串」打分，再上报；
2. **类别型分数必须显式写 `data_type="CATEGORICAL"`**：不写会被当 NUMERIC 解析字符串而报错。

下面这段「客户端初始化 + 降级封装」是三个源文件共用的骨架：**把「有没有密钥」提到最前面**
用 `LANGFUSE_READY` 记住，于是「真连服务」与「只打印报文」两条路径共用同一份业务代码。

In [ ]:
import json
from types import SimpleNamespace

# deepagents 造被测 Agent；langfuse 的 CallbackHandler / @observe 负责把分数挂到 trace 上。
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langfuse import get_client
from langfuse import observe as langfuse_observe
from langfuse.langchain import CallbackHandler

# ---------- 0. 客户端初始化：先判断密钥是否就绪 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,
    )
else:
    langfuse = None


def _observe_local(*dargs, **_dkwargs):
    """降级版 @observe：不上报，只把原函数原样返回（保持课案的代码形状）。"""
    def _deco(func):
        return func

    return dargs[0] if (dargs and callable(dargs[0])) else _deco


observe = langfuse_observe if LANGFUSE_READY else _observe_local


def report_score(**payload) -> None:
    """上报一条分数。

    真连上 Langfuse → 调 create_score 上传；
    密钥为空     → 把「本应发送的报文」打印出来，让学员看清数据结构。
    """
    if LANGFUSE_READY:
        langfuse.create_score(**payload)
    else:
        print("      [降级] 本应上报的 score 报文：" + _payload_text(payload))


def report_score_current_trace(**payload) -> None:
    """上报到「当前所在的 trace」——课案 04 精简版用的就是这个 v4 API：
    在 @observe 函数里不用手动拿 trace_id，SDK 知道你现在在哪个 span 里。
    """
    if LANGFUSE_READY:
        langfuse.score_current_trace(**payload)
    else:
        print("      [降级] 本应上报到「当前 trace」的 score 报文：" + _payload_text(payload))


def _payload_text(payload: dict, limit: int = 120) -> str:
    """把报文压成一行打印；超长的字符串字段（比如整段回答）截断，否则一行几百字看不清。"""
    short = {}
    for key, value in payload.items():
        if isinstance(value, str) and len(value) > limit:
            short[key] = value[:limit] + "…"
        else:
            short[key] = value
    return json.dumps(short, ensure_ascii=False, default=str)

下面造「被测 Agent」和「测试数据集」。数据集故意放三条：

| 测试项 | 期望输出 | 设计意图 |
|---|---|---|
| `1+1等于几` | `2` | 子串匹配能命中，得 1.0 分 |
| `Python的作者是谁` | `Guido van Rossum` | 一般能命中 |
| `用一句话解释什么是 MCP` | 一整句标准答案 | 模型换说法（哪怕答对）必然 0 分 → 引出 LLM 裁判 |

第三条就是「精确字符串匹配的天花板」，第 2、3 节会给出解法。

In [ ]:
# ---------- 1. 演示用的工具与 Agent ----------
@tool
def calculator(expression: str) -> str:
    """执行数学计算。传入数学表达式字符串，如 '1+1'"""
    try:
        return str(eval(expression))
    except Exception:
        return "计算错误"


# 课案这一节写的是 create_deep_agent(model=model, tools=[])；
# 这里挂一个计算器，让「1+1等于几」这条测试项真的走一次工具调用。
agent = create_deep_agent(model=llm, tools=[calculator])


# ---------- 2. 构造测试数据集 ----------
# 课案：先 create_dataset 建空数据集（数据集已存在时返回已有的那个），
#       再 create_dataset_item 逐条添加测试项。
DATASET_NAME = "qa_accuracy"

DATASET_ITEMS = [
    {"input": "1+1等于几", "expected_output": "2"},
    {"input": "Python的作者是谁", "expected_output": "Guido van Rossum"},
    # 第三条故意用「一整句长标准答案」当期望值：子串匹配必然判 0 分，
    # 用来说明「精确字符串匹配太严格」——这正是 05 / 06 要上 LLM-as-Judge 的原因
    {"input": "用一句话解释什么是 MCP",
     "expected_output": "Model Context Protocol（模型上下文协议），是 Anthropic 提出的让大模型连接外部工具与数据的开放标准"},
]


def build_dataset_items():
    """返回可迭代的数据集条目。

    真连上 Langfuse → 走课案流程：create_dataset → create_dataset_item → get_dataset；
    密钥为空       → 用 SimpleNamespace 造出与 dataset.items 同样字段的本地桩数据，
                     这样下面评估循环的代码一个字都不用改。
    """
    if LANGFUSE_READY:
        langfuse.create_dataset(name=DATASET_NAME)
        # 降级时把每一步的报文都打出来，学员照着就能在 UI 上手工复现同一套数据集。
        for item in DATASET_ITEMS:
            langfuse.create_dataset_item(dataset_name=DATASET_NAME, **item)
        dataset = langfuse.get_dataset(DATASET_NAME)
        return dataset.items

    print(f"[降级] 本应发送的 create_dataset 报文：{json.dumps({'name': DATASET_NAME}, ensure_ascii=False)}")
    for item in DATASET_ITEMS:
        payload = {"dataset_name": DATASET_NAME, **item}
        print("[降级] 本应发送的 create_dataset_item 报文："
              + json.dumps(payload, ensure_ascii=False))
# SimpleNamespace 的字段名与 dataset.items 一致，所以下面评估循环一行都不用改。
    return [SimpleNamespace(**item) for item in DATASET_ITEMS]   # 字段名与 dataset.items 一致

评估循环 + `@observe` 打分两部分的功能定义：

In [ ]:
# ---------- 3. 评估循环：跑数据集 → 打分 → 上报 ----------
def demo_dataset_scoring():
    print("\n" + "=" * 72)
    print("一、数据集自动打分：output 里是否包含 expected_output")
    print("=" * 72)

    items = build_dataset_items()
    print()

    for item in items:
        if LANGFUSE_READY:
            langfuse_handler = CallbackHandler()
            # 逐条跑 Agent：真连 Langfuse 时用 handler.last_trace_id，降级时伪造一个可预测的 id。
            output = agent.invoke(
                {"messages": [{"role": "user", "content": item.input}]},
                config={"callbacks": [langfuse_handler]},
            )
            actual = output["messages"][-1].content
            tid = langfuse_handler.last_trace_id          # 这一轮在 Langfuse 里的 trace id
        else:
            output = agent.invoke({"messages": [{"role": "user", "content": item.input}]})
            actual = output["messages"][-1].content
            tid = f"trace-{abs(hash(item.input)) % 100000:05d}"   # 本地伪造一个 id，方便看报文

        # 最朴素的自动打分：期望值是不是回答的子串
        score = 1.0 if str(item.expected_output) in actual else 0.0

            # 打分与上报分开：先算出分数（纯逻辑，好测），再决定是上传还是打印。
        report_score(
            trace_id=tid,
            name="accuracy",
            value=score,
            comment=actual,
        )
        print(f"Q: {item.input} → A: {actual[:40]}... | 得分: {score}   trace={tid}")

    if LANGFUSE_READY:
        langfuse.flush()
    print("\n注意第三条：期望值是一整句标准答案，子串匹配只有模型一字不差复述时才给分；")
    print("模型只要换个说法（哪怕答对了）就拿 0 分 —— 这就是最朴素的自动打分的天花板。")
    print("这类「语义正确但字面不同」的情况需要 LLM 当裁判 —— 见 05 / 06 两个文件。")


# ---------- 4. 在 @observe 里打分：不用手动拿 trace_id ----------
@observe
def answer_with_length_score(question: str) -> str:
    """业务指标打分示例：回复长度是否落在合理区间。"""
    response = llm.invoke(question)
    length = len(response.content)
    report_score_current_trace(
        # score_current_trace 不需要 trace_id：SDK 知道当前代码跑在哪个 span 里。
        name="response_length_ok",
        value=1.0 if 20 <= length <= 200 else 0.0,
        comment=f"回复长度 {length} 字",
    )
    return response.content


@observe
def answer_with_user_feedback(question: str) -> str:
    """用户反馈打分示例：前端点「踩」之后，回调里把反馈上报成类别型分数。"""
    response = llm.invoke(question)
    feedback = "thumb_down"           # 实际来自前端用户的点赞/点踩按钮
    report_score_current_trace(
        name="user_feedback",
        value=feedback,               # 类别型分数直接传字符串
        data_type="CATEGORICAL",      # 不写明 data_type 会被当成 NUMERIC 解析而报错
        comment="用户点了踩：回答太啰嗦",
    )
    return response.content


# 两个业务指标示例：一个数值型（回复长度是否合理）、一个类别型（用户点赞/点踩）。
def demo_observe_scoring():
    print("\n" + "=" * 72)
    print("二、在 @observe 函数里打分：score_current_trace（v4 写法，免 trace_id）")
    print("=" * 72)
    answer_with_length_score("什么是 Function Call？一句话回答")
    print("  业务指标分数已上报（response_length_ok）")
    answer_with_user_feedback("什么是 MCP？一句话回答")
    print("  用户反馈已上报（user_feedback = thumb_down，CATEGORICAL）")
    # 密钥就绪时再 flush 一次，确保刚才两条分数都发出去了。
    if LANGFUSE_READY:
        get_client().flush()

In [ ]:
# 入口：先讲清怎么把 Langfuse 配起来，再走演示。
print("=" * 72)
print("Langfuse ④：评估之「打分（Score）」—— 数据集打分 + 业务指标/用户反馈上报")
print("=" * 72)

if not LANGFUSE_READY:
    # 起始检查：先讲清怎么把 Langfuse 配起来，再走降级演示。
    print("【进入降级演示】Langfuse 密钥为空")
    print("  settings.langfuse_public_key = '' ，settings.langfuse_secret_key = ''")
    print("-" * 72)
    # 这里只讲「缺什么、怎么补」，不重复课案正文；完整的安装说明见 01_追踪_jxsd.py 的文件头。
    print("要看到真实的数据集与分数看板，按课案「安装」一节准备环境：")
    print("  1) git clone https://github.com/langfuse/langfuse.git")
    print("     cd langfuse")
    print("     docker compose up -d          # 启动后访问 http://localhost:3000")
    print("  2) 首次注册的账号即为管理员；新建项目 → Settings → API Keys → 创建密钥")
    print("  3) 写入 F:\\ProGram\\Python_Base\\.env ：")
    print("        LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("        LANGFUSE_SECRET_KEY=sk-lf-...")
    print("        LANGFUSE_HOST=http://localhost:3000    # 本地 docker 部署用这个")
    print("  4) 重新运行本脚本，去 Datasets 看数据集、去 Scores 看分数")
    print("-" * 72)
    # 数据集换成同字段的本地桩数据，所以下面评估循环的代码一个字都不用改。
    print("下面不依赖 Langfuse 服务：数据集换成同字段的本地桩数据，")
    print("Agent 与大模型都真跑，只把「本应上报的分数报文」打印出来。")
    print("=" * 72)
else:
    print("Langfuse 已配置，分数将上传到：", settings.langfuse_host)

# 两节依次演示：数据集自动打分 → 在 @observe 里上报业务指标与用户反馈。
demo_dataset_scoring()
demo_observe_scoring()

print("\n小结：打分就是把「主观好坏」变成可筛选的字段 ——")
print("      trace_id 定位哪一次调用，name 决定看板怎么聚合，comment 留下判断依据。")

### 预期输出

密钥已配好时（本机就是这种），第一节大致是：

```text
Langfuse 已配置，分数将上传到： http://localhost:3001

========================================================================
一、数据集自动打分：output 里是否包含 expected_output
========================================================================
Q: 1+1等于几 → A: ... | 得分: 1.0   trace=...
Q: Python的作者是谁 → A: ... | 得分: 1.0   trace=...
Q: 用一句话解释什么是 MCP → A: ... | 得分: 0.0   trace=...

注意第三条：期望值是一整句标准答案，子串匹配只有模型一字不差复述时才给分；
模型只要换个说法（哪怕答对了）就拿 0 分 —— 这就是最朴素的自动打分的天花板。
这类「语义正确但字面不同」的情况需要 LLM 当裁判 —— 见 05 / 06 两个文件。

========================================================================
二、在 @observe 函数里打分：score_current_trace（v4 写法，免 trace_id）
========================================================================
  业务指标分数已上报（response_length_ok）
  用户反馈已上报（user_feedback = thumb_down，CATEGORICAL）
```

> ⚠️ 每条 `A: ...` 里的回答正文、`trace=...` 的 id、以及第 1/2 条的得分（取决于模型
> 是否把期望值原样写进回答）都由模型与运行决定、每次可能不同。**能确定的是**：
> 「MCP 一长句标准答案」这条几乎必然 0 分 —— 这正是本节要展示的「子串匹配天花板」。
> 另外**条目数会随重跑累积**：`create_dataset_item` 是追加式的，上面只画了 3 条，
> 实际会打印 `qa_accuracy` 数据集里**全部**条目（本机这次远多于 3 条）。

## 2. RAG 评估与 Agent 评估：四项指标（Experiment）

RAG 系统有两个会出错的环节，所以指标也分两组：

| 指标组 | 指标 | 一句话 | 计算方法 |
|---|---|---|---|
| 检索质量 | Context Precision 上下文相关性 | 相关上下文是不是排在前面？ | LLM 逐条判相关，算比例 |
| 检索质量 | Context Recall 上下文召回率 | 标准答案要的信息上下文给全了吗？ | 把标准答案拆事实点，逐个判能否从上下文推断 |
| 生成质量 | Faithfulness 忠实度 | 回答里每句话都能从上下文推出来吗？ | 拆陈述，判可推断数 / 总数 |
| 生成质量 | Answer Relevancy 答案相关性 | 回答有没有直接答用户的问题？ | 由回答反推问题，算语义相似度均值 |

### 2.1 课案原版：`run_experiment` 最短实现

Langfuse 的 `run_experiment(name, data, task, evaluators)` 一个函数把
「批量跑任务 + 逐条评分」打包了：`task` 是被测流程，`evaluators` 是评分函数列表。

In [ ]:
from langchain.chat_models import init_chat_model
from langfuse import Langfuse
from config import settings

lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    host=settings.langfuse_host,
)

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

In [ ]:
# ---------- 1. 准备测试用例（也可换成控制台 Dataset 里的 items） ----------
data = [
    {"input": {"question": "LangChain 是什么？"},
     "expected_output": {"keywords": "LLM 应用开发框架"}},
    {"input": {"question": "LangGraph 是什么？"},
     "expected_output": {"keywords": "智能体图编排框架"}},
]

# ---------- 2. 定义被测任务（换成你的 RAG/Agent 流程） ----------
def my_rag_task(*, item, **kwargs):
    """被评估的任务：输入问题，返回回答（这里用裸 LLM 模拟）"""
    question = item["input"]["question"]
    answer = llm.invoke(question).content
    return {"answer": answer}

# ---------- 3. 定义评分函数 ----------
def relevance_scorer(*, input, output, expected_output, **kwargs):
    """关键词命中率：expected 的关键词是否出现在回答里"""
    keywords = expected_output["keywords"]
    answer = output["answer"]
    hit = all(k in answer for k in keywords.split())
    return {"name": "keyword_hit", "value": 1.0 if hit else 0.0}

In [ ]:
# ---------- 4. 跑实验 ----------
try:
    result = lf.run_experiment(
        name="baseline-v1",
        data=data,
        task=my_rag_task,
        evaluators=[relevance_scorer],
    )
    print("实验完成！结果概览：", result)
except Exception as e:
    print("实验执行失败（检查 Langfuse 密钥是否配置）：", e)
finally:
    lf.flush()

# RAG 评估进阶：ragas 库提供 faithfulness / answer_relevancy 等标准指标，
# 把 ragas 的 score 函数包成 evaluator 传入 run_experiment 即可。

### 预期输出

```text
实验完成！结果概览： <langfuse.experiment.ExperimentResult object at 0x...>
```

> ⚠️ 那串对象地址每次运行都不同（`0x...`）。关键是「实验完成」四个字——
> 说明 `run_experiment` 把两条用例各跑了一遍并打了 `keyword_hit` 分。
> 结果对象本身可用 `result.format()` 打印成可读的表格（原版没打印，留给你试）。

### 2.2 完整版：本地 LLM-as-Judge 四项指标

原版只演示了 `run_experiment` 的调用形状，评分还是「关键词命中」这种玩具。
完整版把课案的四个 **ragas 指标**真正落地。但本机**没装 ragas**（装 ragas 要拉
`langchain-community` 全量依赖），所以走**同一方法论的本地 LLM-as-Judge 近似版**：
把「拆事实点 / 拆陈述 / 反推问题」这些裁判提示词手写一遍，四项数值照常真算。

In [ ]:
import json
import re
from types import SimpleNamespace

# langfuse 只负责「分数往哪去」；ragas 缺失时下面走本地等价实现，四项数值照常算得出。
from langchain.chat_models import init_chat_model
from langfuse import Langfuse
from config import settings

# ---------- 0. 可选依赖：ragas 没装也要能 import 本文件 ----------
try:
    from ragas.llms import llm_factory                     # noqa: F401
    from ragas.metrics.collections import (                # noqa: F401
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        Faithfulness,
    )
    RAGAS_AVAILABLE = True
    RAGAS_IMPORT_ERROR = ""
except ImportError as exc:                                 # 缺包：记下原因，走本地近似实现
    RAGAS_AVAILABLE = False
    RAGAS_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"

# ---------- 1. 客户端与模型 ----------
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        host=settings.langfuse_host,
    )
else:
    langfuse = None

# 生成回答用的业务模型；评测（裁判）模型另有一份，见 build_eval_client。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def report_score(**payload) -> None:
    """上报一条分数：连上 Langfuse 就真上传，否则打印报文（降级演示）。"""
    if LANGFUSE_READY:
        langfuse.create_score(**payload)
    else:
        short = {k: (v[:120] + "…" if isinstance(v, str) and len(v) > 120 else v)
                 for k, v in payload.items()}
        print("        [降级] 本应上报的 score 报文：" + json.dumps(short, ensure_ascii=False, default=str))


def build_eval_client():
    """构造评测用的 OpenAI 兼容客户端。

    课案原文：
        eval_client = AsyncOpenAI(api_key=settings.dashscope_api_key,
                                  base_url=settings.dashscope_base_url)
    本机 dashscope 两项为空 → 成对降级到 settings.api_key / settings.base_url。
    """
    if settings.dashscope_api_key:
        return settings.dashscope_api_key, settings.dashscope_base_url, "DashScope（课案原配）"
    return settings.api_key, settings.base_url, f"{settings.model_name}（项目通用大模型，降级）"

数据集（课案原文三条，`contexts` 一字未改，这样算出来的分数才能和课案对得上）：

In [ ]:
# ---------- 2. 构造 RAG 评估数据集（课案原文三条，上下文一字未改） ----------
DATASET_NAME = "rag_evaluation"

# 课案原文三条测试项，contexts 一字未改 —— 这样算出来的分数才能和课案对得上。
DATASET_ITEMS = [
    {
        "input": "Transformer 模型的核心机制是什么？",
        "expected_output": "Transformer 的核心是自注意力（Self-Attention）机制，"
                           "它允许模型在处理每个词时关注输入序列中的所有位置，"
                           "从而捕获长距离依赖关系。",
        "metadata": {
            "contexts": [
                "Transformer 架构由 Vaswani 等人在 2017 年提出，完全基于注意力机制，摒弃了循环和卷积结构。",
                "自注意力机制是 Transformer 的核心，通过 Query、Key、Value 三个矩阵计算序列中每个位置与其他位置的关联权重。",
                "多头注意力将注意力计算拆分到多个子空间，使模型能同时关注不同位置的不同特征表示。",
            ]
        },
    },
    # 第 2 条的 contexts 已完整覆盖标准答案，用来验证「检索给全了」时召回率应接近 1。
    {
        "input": "什么是 RAG？它解决了什么问题？",
        "expected_output": "RAG（检索增强生成）是一种将信息检索与文本生成相结合的技术架构，"
                           "主要解决了大语言模型的幻觉问题和知识时效性问题。",
        "metadata": {
            "contexts": [
                "RAG（Retrieval-Augmented Generation）由 Facebook AI 在 2020 年提出，"
                "工作流程：用户提问 → 从知识库检索相关文档 → 将检索结果作为上下文注入 Prompt → LLM 生成答案。",
                "RAG 解决的两大核心问题：1）幻觉——模型编造不存在的事实；"
                "2）知识时效性——训练数据截止日期后的事件模型无法知晓。",
                "RAG 的优势：无需微调模型即可接入最新知识，答案可溯源至检索到的文档片段。",
            ]
        },
    },
    # 第 3 条考跨句推断：单个事实点要能把三条上下文里的信息拼起来才推得出。
    {
        "input": "向量数据库在 RAG 中起什么作用？",
        "expected_output": "向量数据库在 RAG 中负责高效存储和检索文档的向量嵌入，"
                           "是 RAG 检索阶段的核心基础设施。",
        "metadata": {
            "contexts": [
                "向量数据库将文本通过嵌入模型转换为高维向量，"
                "查询时用同样的嵌入模型将问题转为向量，通过余弦相似度等度量找到最相近的文档。",
                "常见向量数据库：Chroma、Pinecone、Weaviate、Milvus、Qdrant。",
                "在 RAG 流水线中，向量数据库处于检索阶段 —— "
                "接收用户查询向量，返回 Top-K 相关文档片段供 LLM 参考。",
            ]
        },
    },
]


def build_dataset_items():
    """真连上 Langfuse 就走课案流程；否则用同字段的本地桩数据，让评估循环代码不变。"""
    if LANGFUSE_READY:
        # 真连上就走课案流程（建集 → 逐条加 → get 回来），字段名与 dataset.items 完全一致。
        langfuse.create_dataset(name=DATASET_NAME)
        for item in DATASET_ITEMS:
            langfuse.create_dataset_item(dataset_name=DATASET_NAME, **item)
        return langfuse.get_dataset(DATASET_NAME).items

    print(f"[降级] 本应创建 Langfuse 数据集 {DATASET_NAME}，共 {len(DATASET_ITEMS)} 条测试项")
    for item in DATASET_ITEMS:
        print(f"        · {item['input']}  （contexts {len(item['metadata']['contexts'])} 条）")
    return [SimpleNamespace(**item) for item in DATASET_ITEMS]

课案原文路径（装了 ragas 才走）与降级路径（本机走这条）的定义：

In [ ]:
# ---------- 3. 课案原文路径：ragas 官方四项指标 ----------
def build_ragas_metrics(eval_model_name, eval_client):
    """按课案原文创建四项评估指标实例。

        evaluator_llm = llm_factory(model, client=eval_client, max_tokens=8192)
        cp_metric = ContextPrecision(llm=evaluator_llm)
        cr_metric = ContextRecall(llm=evaluator_llm)
        faith_metric = Faithfulness(llm=evaluator_llm)
        ar_metric = AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_emb, strictness=1)

    strictness 控制「从回答反推生成多少个问题」来检测相关性：1 = 只生成 1 个。
    """
    from ragas.embeddings import OpenAIEmbeddings

    evaluator_llm = llm_factory(eval_model_name, client=eval_client, max_tokens=8192)
    # 课案的 embedding 模型是阿里百炼的 text-embedding-v4；
    # 降级到通用大模型时通常没有 embedding 接口，AnswerRelevancy 需要句向量，
    # 这时候只能退回本地近似实现（见下面 local_answer_relevancy）。
    evaluator_emb = OpenAIEmbeddings(client=eval_client, model="text-embedding-v4")
    return {
        "Context Precision": ContextPrecision(llm=evaluator_llm),
        "Context Recall": ContextRecall(llm=evaluator_llm),
        "Faithfulness": Faithfulness(llm=evaluator_llm),
        "Answer Relevancy": AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_emb, strictness=1),
    }


# ---------- 4. 降级路径：本地 LLM-as-Judge 近似版四项指标 ----------
# 说明：ragas 官方实现各有成套的提示词与多次采样，下面是**同一方法论的教学简化版**，
#       目的是没装 ragas 时也能看到四个真实数值。口径与 ragas 不完全一致，
#       两个工具的分数不要直接横向比较。
def judge(prompt: str) -> dict:
    """通用裁判调用：把提示词发给大模型，要求它只回 JSON，再解析成 dict。

    健壮性处理（线上评估脚本必须做）：
        - 模型爱把 JSON 包在 ```json ``` 里 → 用正则把第一段 {...} 抠出来
        - 网络抖动 / 模型抽风 → 捕获异常并返回 {}，让调用方走兜底分数，
          绝不能因为一条评估项失败就把整个评估任务挂掉
    """
    try:
        text = llm.invoke(prompt).content
    except Exception as exc:
        print(f"        [警告] 裁判模型调用失败：{type(exc).__name__}: {exc}")
        return {}
    match = re.search(r"\{.*\}", text, re.S)      # 贪婪匹配到最后一个 }，容错 ```json 包裹
    if not match:
        print(f"        [警告] 裁判模型没有返回 JSON：{text[:80]}…")
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        print(f"        [警告] 裁判返回的 JSON 解析失败：{exc}")
        return {}


def local_context_precision(question: str, contexts: list[str]) -> float:
    """上下文相关性：相关上下文所占比例（0~1）。"""
    numbered = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(contexts))
    data = judge(
    # 一次调用问完所有上下文：让裁判按顺序回 true/false 数组。
        f"问题：{question}\n\n检索到的上下文：\n{numbered}\n\n"
        "请逐条判断每条上下文是否与「回答问题」直接相关（只要含有能回答问题的信息就算相关）。\n"
        '只输出 JSON：{"relevant": [true, false, ...]}，数组长度必须等于上下文条数。'
    )
    flags = data.get("relevant")
    if not isinstance(flags, list) or not flags:
        return 0.0
    flags = [bool(x) for x in flags[: len(contexts)]]
    return sum(flags) / len(contexts)


def local_context_recall(question: str, contexts: list[str], reference: str) -> float:
    """上下文召回率：标准答案的事实点被上下文覆盖的比例（0~1）。"""
    data = judge(
        f"问题：{question}\n\n标准答案：{reference}\n\n"
        f"检索到的上下文：\n" + "\n".join(f"- {c}" for c in contexts) + "\n\n"
        "请把标准答案拆解成若干独立事实点，再逐个判断该事实点能否从上面的上下文中推断出来。\n"
        '只输出 JSON：{"total": 事实点总数, "covered": 能被上下文覆盖的事实点数}'
    )
    # 裁判回非数值一律判 0 分：兜底优先于「猜一个分数」，不能让一条用例挂掉整轮评估。
    total, covered = data.get("total"), data.get("covered")
    if not isinstance(total, (int, float)) or not total:
        return 0.0
    return max(0.0, min(1.0, float(covered or 0) / float(total)))


def local_faithfulness(question: str, answer: str, contexts: list[str]) -> float:
    """忠实度：回答里的陈述能被上下文支撑的比例（0~1），越低说明越可能在编。"""
    data = judge(
        f"问题：{question}\n\n检索到的上下文：\n" + "\n".join(f"- {c}" for c in contexts) +
        f"\n\n模型生成的回答：{answer}\n\n"
        "请把回答拆成若干条独立陈述，逐条判断能否由上面的上下文推断出来。\n"
        '只输出 JSON：{"total": 陈述总数, "supported": 能被上下文支撑的陈述数}'
    )
    # 与召回率同构：先把回答拆成陈述，再逐条判断能否由上下文推出。
    total, supported = data.get("total"), data.get("supported")
    if not isinstance(total, (int, float)) or not total:
        return 0.0
    return max(0.0, min(1.0, float(supported or 0) / float(total)))


def local_answer_relevancy(question: str, answer: str, strictness: int = 1) -> float:
    """答案相关性：由回答反推问题，再算这些问题与原问题的语义相似度均值（0~1）。

    真 ragas 用 embedding 算余弦相似度；这里没有句向量模型，
    改让同一个裁判模型在生成反向问题的同时给出相似度评分（一次调用完成）。
    """
    data = judge(
        f"原始问题：{question}\n\n模型回答：{answer}\n\n"
        f"请基于这个回答，反向推断出 {strictness} 个「它最可能是在回答的问题」，"
        "并给每个反向问题与原问题的语义相似度打分（0~1）。\n"
        '只输出 JSON：{"questions": ["反向问题1"], "similarities": [0.0]}'
    )
    # 真 ragas 用 embedding 算余弦相似度；这里让裁判一次调用同时给出反向问题与相似度。
    sims = data.get("similarities")
    if not isinstance(sims, list) or not sims:
        return 0.0
    values = [float(s) for s in sims if isinstance(s, (int, float))]
    return sum(values) / len(values) if values else 0.0


def local_metrics(question: str, answer: str, contexts: list[str], reference: str) -> dict:
    """四项指标一次算完，返回 {指标名: 分数}。"""
    return {
        "Context Precision": local_context_precision(question, contexts),
        "Context Recall": local_context_recall(question, contexts, reference),
        "Faithfulness": local_faithfulness(question, answer, contexts),
        "Answer Relevancy": local_answer_relevancy(question, answer, strictness=1),
    }

主评估流程（逐条拼上下文 → 生成回答 → 四项打分 → 上报）：

In [ ]:
# ---------- 5. 主评估流程 ----------
def run_evaluation():
    eval_model_name = settings.model_name
    eval_key, eval_base, eval_source = build_eval_client()
    eval_client = None
    metrics = None

    # 装了 ragas 用官方四项指标；没装走本地近似实现，两者口径不同，分数不要横向比较。
    if RAGAS_AVAILABLE:
        from openai import AsyncOpenAI
        eval_client = AsyncOpenAI(api_key=eval_key, base_url=eval_base)
        metrics = build_ragas_metrics(eval_model_name, eval_client)
        print(f"评估器：ragas 官方指标，裁判模型 {eval_model_name} @ {eval_source}")
    else:
        print("评估器：本地 LLM-as-Judge 近似实现（未安装 ragas）")
        print(f"        裁判模型 {settings.model_name} @ {eval_source}")

    print("\n" + "=" * 72)
    print("逐条评估：拼上下文 → 生成回答 → 四项指标打分 → 上报")
    print("=" * 72)

    # 两种数据源的字段名对齐，所以下面这段评估循环一行都不用改。
    items = build_dataset_items()
    for idx, item in enumerate(items, start=1):
        contexts = item.metadata.get("contexts", [])
        context_text = "\n".join(f"- {ctx}" for ctx in contexts)
        prompt = (
            f"请根据以下参考资料回答问题。\n\n"
            f"参考资料：\n{context_text}\n\n"
            f"问题：{item.input}"
        )

        print(f"\n[{idx}/{len(items)}] Q: {item.input}")
        # 课案这里用的是 agent.invoke（DeepAgents）；RAG 的「增强生成」本来就是
        # 「上下文拼进 Prompt → 模型作答」这一步，这里直接用大模型调用表示自己系统流程。
        answer = llm.invoke(prompt).content
        print(f"        A: {answer[:80]}…")

        if RAGAS_AVAILABLE:
            # 课案的写法：四个指标各自 .score(...)，取 .value
            cp = metrics["Context Precision"].score(
                user_input=item.input, reference=item.expected_output, retrieved_contexts=contexts,
            ).value
            cr = metrics["Context Recall"].score(
                user_input=item.input, retrieved_contexts=contexts, reference=item.expected_output,
            ).value
            faith = metrics["Faithfulness"].score(
                user_input=item.input, response=answer, retrieved_contexts=contexts,
            ).value
            ar = metrics["Answer Relevancy"].score(
                user_input=item.input, response=answer,
            ).value
        else:
            scores = local_metrics(item.input, answer, contexts, item.expected_output)
            cp, cr, faith, ar = (scores["Context Precision"], scores["Context Recall"],
                                 scores["Faithfulness"], scores["Answer Relevancy"])

        # 真实环境里 tid 来自 langfuse_handler.last_trace_id；这里用可预测的假 id 便于对照报文。
        tid = "trace-rag-%03d" % idx          # 真实环境里是 langfuse_handler.last_trace_id

        # 课案：四项分数逐个上报到同一条 trace 上，name 就是指标名
        report_score(trace_id=tid, name="Context Precision", value=cp, comment=f"contexts={len(contexts)}条")
        report_score(trace_id=tid, name="Context Recall", value=cr, comment=f"contexts={len(contexts)}条")
        report_score(trace_id=tid, name="Faithfulness", value=faith, comment="回答是否忠于上下文")
        report_score(trace_id=tid, name="Answer Relevancy", value=ar, comment="回答是否切题")

        # 一行打完四个分数，方便直接和课案截图里的数值对照。
        print(f"        CP={cp:.2f}  CR={cr:.2f}  Faith={faith:.2f}  AR={ar:.2f}")
        print(f"        Trace: {tid}")
        print("-" * 50)

    if LANGFUSE_READY:
        langfuse.flush()

In [ ]:
# 入口：先讲清本机缺什么（ragas / Langfuse / DashScope 三处），再真跑评估。
print("=" * 72)
print("RAG 评估四项指标：Context Precision / Context Recall / Faithfulness / Answer Relevancy")
print("=" * 72)

if not RAGAS_AVAILABLE:
    # 依赖缺失只提示、不退出：本地实现同样能算出四个数值。
    print("\n【依赖未安装】没找到 ragas，本文件按规范用 try/except 兜住，不会报错退出。")
    print(f"  import 失败原因：{RAGAS_IMPORT_ERROR}")
    print("  安装命令（课案给的）：")
    print('      uv add langchain-openai ragas "langchain-community==0.3.30"')
    print("  装完后重跑本脚本，就会自动切换到 ragas 官方指标。")

if not LANGFUSE_READY:
    # 密钥缺失同理：只影响「上报」这一步，不影响打分。
    print("\n【进入降级演示】Langfuse 密钥为空")
    print("  settings.langfuse_public_key = '' ，settings.langfuse_secret_key = ''")
    print("-" * 72)
    print("要看到真实的评估看板，按课案「安装」一节准备环境：")
    print("  1) git clone https://github.com/langfuse/langfuse.git")
    print("     cd langfuse")
    print("     docker compose up -d          # 启动后访问 http://localhost:3000")
    print("  2) 首次注册的账号即为管理员；新建项目 → Settings → API Keys → 创建密钥")
    print("  3) 写入 F:\\ProGram\\Python_Base\\.env ：")
    print("        LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("        LANGFUSE_SECRET_KEY=sk-lf-...")
    print("        LANGFUSE_HOST=http://localhost:3000    # 本地 docker 部署用这个")
    print("  4) 重新运行本脚本，四项指标会出现在 trace 详情与 Scores 看板里")
    print("-" * 72)
    print("下面不依赖 Langfuse 服务：数据集用同字段的本地桩数据，")

if not settings.dashscope_api_key:
    # 评测模型降级说明：key 与 base_url 必须成对降级，只换其中一个会 401。
    print("\n【评测模型降级】settings.dashscope_api_key / dashscope_base_url 为空")
    print("  课案用 DashScope（阿里百炼）当裁判：它有 text-embedding-v4 供 AnswerRelevancy")
    print("  算句向量，而且评测模型与业务模型分开、避免自评偏袒。")
    print(f"  这里成对降级到 settings.api_key / settings.base_url（当前模型 {settings.model_name}）。")
    print("  想用回课案原配：去阿里百炼控制台建 key，然后写进 .env ：")
    print("        DASHSCOPE_API_KEY=sk-...")
    print("        DASHSCOPE_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1")

if not LANGFUSE_READY:
    # 三条降级说明都打完了，下面开始真跑。
    print("\n生成回答与四项打分全部真跑，只把「本应上报的分数报文」打印出来。")
    print("=" * 72)

run_evaluation()

# 收尾小结：四项分数挂在同一条 trace 上，才能在看板里按版本、按时间对比。
print("\n小结：检索看 CP / CR（找得准不准、全不全），生成看 Faithfulness / AR（编没编、切不切题）；")
print("      四项分数都挂在同一条 trace 上，才能在 Langfuse 看板里按版本、按时间对比。")

### 预期输出

```text
========================================================================
RAG 评估四项指标：Context Precision / Context Recall / Faithfulness / Answer Relevancy
========================================================================
【依赖未安装】没找到 ragas，本文件按规范用 try/except 兜住，不会报错退出。
  评估器：本地 LLM-as-Judge 近似实现（未安装 ragas）
  逐条评估：拼上下文 → 生成回答 → 四项指标打分 → 上报
========================================================================
[1/N] Q: Transformer 模型的核心机制是什么？
        CP=...  CR=...  Faith=...  AR=...
[2/N] Q: 什么是 RAG？它解决了什么问题？
        CP=...  CR=...  Faith=...  AR=...
[3/N] Q: 向量数据库在 RAG 中起什么作用？
        CP=...  CR=...  Faith=...  AR=...
小结：检索看 CP / CR（找得准不准、全不全），生成看 Faithfulness / AR（编没编、切不切题）；
```

> ⚠️ 每一行的 `A: ...`、四个分数、以及「裁判模型 @ 哪个源」都由模型决定、每次不同。
> `N` 是 `rag_evaluation` 数据集里累积的条目数（追加式，重跑一次多 3 条，本机这次 18 条）。
> 源文件头里记的本机实测（未装 ragas，3 条用例）是：CP/CR/Faith/AR 大致
> `1.00/0.67/1.00/1.00`、`0.67/1.00/0.71/0.90`、`0.67/1.00/0.80/0.85` 这个量级——
> **别逐字比对具体分数**，要看的是「四个分数都能算出来、且分属两个维度」这件事。

## 3. Agent 评估四指标：把评估对象从「回答」换成「行为」

这一节和 RAG 评估最大的不同：评估对象从「回答」变成了「**行为**」——
Agent 调了哪些工具、有没有跑题、目标达成没有。

| 小节 | 指标 | 输入 | 打分方式 | 值域 |
|---|---|---|---|---|
| 主题一致性 | TopicAdherence | 消息轨迹 + reference_topics | LLM 判定回答/拒绝/越界 | P / R / F1 |
| 工具调用准确率 | ToolCallAccuracy | 轨迹工具调用 + 参考调用 | 名称+参数完全匹配（二元） | 0 或 1 |
| 工具调用 F1 | ToolCallF1 | 同上 | 无序匹配算精确率/召回率/F1 | 0~1 |
| 智能体目标准确率 | AgentGoalAccuracy | 轨迹 + reference（可选） | LLM 裁判判定目标是否达成 | 0 或 1 |

本机没装 ragas（这四个指标类在 ragas 里），所以按课案给的公式**自己实现**。
好处：四个评估函数都是纯数据进、纯数据出，**不依赖 Langfuse 也能跑**。

### 3.1 消息结构体（ragas 缺失时的本地等价实现）

课案原文 `from ragas.messages import HumanMessage, AIMessage, ToolMessage, ToolCall`。
本机没装 ragas，于是定义一套**字段完全相同**的轻量 `@dataclass`；装了 ragas 的机器会自动
改用官方那套，函数体无需改动。

In [ ]:
import json
import re
from collections import Counter
from dataclasses import dataclass, field
from types import SimpleNamespace

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage as LCAI
from langchain_core.messages import HumanMessage as LCHuman
from langchain_core.messages import ToolMessage as LCTool
from langchain_core.tools import tool
from langfuse import Langfuse
from config import settings

# ---------- 0. 可选依赖：ragas 装了就用它的消息结构体，没装就用本地等价结构体 ----------
try:
    from ragas.messages import AIMessage, HumanMessage, ToolCall, ToolMessage  # noqa: F401
    MESSAGE_SOURCE = "ragas.messages（课案原配）"
except ImportError:
    # 下面四个类的字段名、字段顺序都对着 ragas.messages 抄，只是没有 ragas 的额外方法
    @dataclass
    class ToolCall:                      # noqa: D101  （对应 ragas.messages.ToolCall）
        name: str                        # 工具名
        args: dict = field(default_factory=dict)   # 参数；用 default_factory 而不是 {} 避免可变默认值共享

    @dataclass
    class HumanMessage:                  # noqa: D101
        content: str                     # 用户说的话

    @dataclass
    class AIMessage:                     # noqa: D101
        content: str = ""
        tool_calls: list | None = None    # 允许为空：模型可以「只说话不调工具」

    @dataclass
    class ToolMessage:                   # noqa: D101
        content: str = ""                # 工具的返回值

    MESSAGE_SOURCE = "本地等价结构体（未安装 ragas）"

In [ ]:
# ---------- 1. 客户端与模型 ----------
# 与 01~05 保持一致：密钥为空就不实例化真客户端，免得 SDK 往 stderr 刷认证错误。
# 四个指标本身不依赖 Langfuse，所以降级只影响「上报分数」这一步，不影响算分。
LANGFUSE_READY = bool(settings.langfuse_public_key and settings.langfuse_secret_key)

if LANGFUSE_READY:
    langfuse = Langfuse(
        public_key=settings.langfuse_public_key,
        secret_key=settings.langfuse_secret_key,
        # 密钥为空就置 None；四个指标本身不依赖 Langfuse，所以不影响算分。
        host=settings.langfuse_host,
    )
else:
    langfuse = None

# 一个模型干两件事：当被测 Agent 的大脑，也当评估用的裁判（LLM-as-Judge）。
# 生产里最好拆成两个模型，避免「自己评自己」偏袒 —— 05 文件里换 DashScope 就是这个原因。
# 一个模型干两件事：当被测 Agent 的大脑，也当评估用的裁判。生产里建议拆成两个模型。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def report_score(**payload) -> None:
    """上报一条分数：连上 Langfuse 就真上传，否则打印报文（降级演示）。

    用 **payload 而不是固定形参，是因为 Langfuse 的 create_score 字段很多
    （trace_id / name / value / comment / data_type / observation_id ...），
    写死形参会逼着调用方填一堆 None。
    """
    if LANGFUSE_READY:
        langfuse.create_score(**payload)
    else:
        # 打印前先截断长字符串：comment 里可能塞着整段模型回答，不截断会刷屏
        short = {k: (v[:100] + "…" if isinstance(v, str) and len(v) > 100 else v)
                 for k, v in payload.items()}
        print("        [降级] 本应上报的 score 报文：" + json.dumps(short, ensure_ascii=False, default=str))

### 3.2 工具与 Agent

三个工具对应三种典型行为：纯计算、外部检索、外部查询。注意每个工具的 docstring
**就是模型看到的 description** —— 写得越清楚，选错工具的概率越低。

还设了一个 `AGENT_STEP_LIMIT`：模型偶尔会陷入「同一句话反复调同一个工具」的循环，
不设上限一条用例能跑几十分钟、刷几万行。触到上限就把已跑出的轨迹记为「未完成」。

In [ ]:
# ---------- 2. 定义工具与 Agent（与课案一致） ----------
@tool
def calculator(expression: str) -> str:
    """执行数学计算。传入数学表达式字符串，如 '15*8+23'"""
    try:
        return str(eval(expression))       # 教学演示用；生产环境请换成安全求值
    except Exception:
        return "计算错误"                   # 工具内部兜错：宁可回一句错误文本，也别抛异常打断 Agent


@tool
def web_search(query: str) -> str:
    """搜索互联网获取最新信息。传入搜索关键词"""
    # 固定的假结果：教学脚本不真的联网，这样每次跑分数才可复现
    return f"关于'{query}'的搜索结果：Python 最新稳定版本为 3.13，发布于 2024 年 10 月。"


@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气信息"""
    return f"{city}今天晴，气温 30°C，湿度 45%。"


agent = create_deep_agent(model=llm, tools=[calculator, web_search, get_weather])

# DeepAgents 自带文件系统 / 待办 / 子 Agent 等内部工具，
# 评估工具调用时只关心我们自己定义的那三个，其余全部过滤掉
CUSTOM_TOOLS = {"calculator", "web_search", "get_weather"}

# 单次评估最多允许图执行多少步（LangGraph 的 recursion_limit）。
# 为什么评估脚本必须设它：模型偶尔会陷入「同一句话反复调同一个工具」的循环，
# 实测不设上限时一条用例能连续调用几百次，既跑几十分钟、又刷屏几万行。
# 触到上限时保留已经跑出来的轨迹，把这条用例记为「未完成」，继续下一条。
AGENT_STEP_LIMIT = 30


def run_agent(question: str) -> tuple[list, bool]:
    """跑一次 Agent，返回（消息列表, 是否被步数上限截断）。

    用 stream(stream_mode="values") 而不是 invoke：每一步都能拿到完整状态，
    万一触到 recursion_limit 抛错，也已经把此前的消息攒下来了（invoke 会直接丢状态）。
    """
    messages, truncated = [], False
    try:
        # stream_mode="values" 每个 chunk 都是「当前完整状态」，
        # 所以循环结束时 messages 天然就是最后一步的完整消息列表，不用自己拼接
        for chunk in agent.stream(
            {"messages": [{"role": "user", "content": question}]},
            config={"recursion_limit": AGENT_STEP_LIMIT},
            stream_mode="values",
        ):
            messages = chunk["messages"]
    except Exception as exc:
        # 典型是 langgraph 的 GraphRecursionError；网络/模型侧报错也在这里兜住，
        # 绝不让一条评估用例把整轮评估带崩
        truncated = True
        print(f"        [提示] 本次执行被中断（{type(exc).__name__}），"
              f"按已完成的 {len(messages)} 条消息继续评估")
    return messages, truncated

### 3.3 消息转换与裁判工具

把 LangChain 消息转成评估轨迹时**必须过滤掉 DeepAgents 内部工具**
（`write_todos` / `read_file` 这类），否则工具调用准确率永远算不对。

In [ ]:
# ---------- 3. 消息转换：LangChain 消息 → 评估用的消息轨迹 ----------
def to_eval_messages(lc_messages):
    """把 LangChain 消息列表转成评估用的轨迹（只保留自定义工具的调用与返回）。

    为什么必须过滤：DeepAgents 内部会调 write_todos / read_file 这类工具，
    它们不属于「被测行为」。不过滤的话，工具调用准确率会永远算不对。
    """
    messages = []
    for msg in lc_messages:
        # 三种 LangChain 消息各自映射到一种 ragas 消息，一一对应
        if isinstance(msg, LCHuman):
            messages.append(HumanMessage(content=str(msg.content)))
        elif isinstance(msg, LCAI):
            calls = None
            raw_calls = getattr(msg, "tool_calls", None) or []
            # 只留 CUSTOM_TOOLS 里的调用；内部工具（write_todos 等）在这里被滤掉
            kept = [ToolCall(name=tc["name"], args=tc.get("args", {}))
                    for tc in raw_calls if tc.get("name") in CUSTOM_TOOLS]
            calls = kept or None            # 全是内部工具就置 None，避免产生空调用
            messages.append(AIMessage(content=str(msg.content or ""), tool_calls=calls))
        elif isinstance(msg, LCTool):
            # 工具的返回消息只在「它属于自定义工具」时才保留，和上面的过滤保持对称
            if getattr(msg, "name", None) in CUSTOM_TOOLS:
                messages.append(ToolMessage(content=str(msg.content)))
    return messages


def format_trajectory(messages) -> str:
    """把轨迹拍平成文本，给 LLM 裁判看。

    裁判只会读文本，所以这里把结构化消息渲染成「[用户]/[助手]/[工具返回]」三类行，
    模型才能顺着时间线判断「答了什么、拒了什么、有没有越界」。
    """
    lines = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            lines.append(f"[用户] {msg.content}")
        elif isinstance(msg, AIMessage):
            # 一条 AIMessage 可能既调了工具又说了话，两段都要输出
        # 一条 AIMessage 可能既调了工具又说了话，两段都要输出给裁判看。
            if msg.tool_calls:
                calls = ", ".join(f"{tc.name}({json.dumps(tc.args, ensure_ascii=False)})"
                                  for tc in msg.tool_calls)
                lines.append(f"[助手] 调用工具 → {calls}")
            if msg.content:
                lines.append(f"[助手] {msg.content}")
        elif isinstance(msg, ToolMessage):
            lines.append(f"[工具返回] {msg.content}")
    return "\n".join(lines)


def collect_actual_tool_calls(messages) -> list:
    """从轨迹里按顺序取出所有自定义工具调用。

    顺序有意义的两个理由：① ToolCallAccuracy(strict_order=True) 要比对顺序；
    ② F1 虽然是无序匹配，但保留顺序方便出问题时人工核对。
    """
    calls = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            calls.extend(msg.tool_calls)      # 一条消息里可能并列多个 tool_calls，用 extend
    return calls


def format_calls(calls: list, limit: int = 6) -> str:
    """把工具调用列表压成一行可读文本；重复调用显示成 name×次数。

    评估输出必须能看懂：模型陷入循环时可能有几百次调用，原样打印会刷屏几万字符。
    """
    if not calls:
        return "（没有调用自定义工具）"
    # 先按「名称+参数」计数：同一个工具被调 200 次只占一项
    counter = Counter(f"{tc.name}({json.dumps(tc.args, ensure_ascii=False)})" for tc in calls)
    parts = [f"{text}×{n}" if n > 1 else text for text, n in counter.items()]
    if len(parts) > limit:
        # 超过 limit 就只留前 limit 项，末尾补一句总次数，保证信息不丢
        parts = parts[:limit] + [f"…共 {sum(counter.values())} 次调用"]
    return ", ".join(parts)

In [ ]:
# ---------- 4. LLM 裁判的公共工具 ----------
def judge(prompt: str) -> dict:
    """把提示词发给大模型，要求只回 JSON，解析成 dict。

    评估脚本必须扛得住模型的自由发挥：这里用正则抠出第一段 {...}，
    解析失败就返回 {} 让调用方走兜底分，绝不让一条评估项挂掉整个评估。
    """
    try:
        text = llm.invoke(prompt).content
    except Exception as exc:
        print(f"        [警告] 裁判模型调用失败：{type(exc).__name__}: {exc}")
        return {}
    match = re.search(r"\{.*\}", text, re.S)   # re.S 让 . 能跨行匹配，JSON 常被模型换行排版
    if not match:
        print(f"        [警告] 裁判没返回 JSON：{text[:80]}…")
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        # 解析失败时返回空 dict，调用方会走兜底分 —— 评估脚本绝不能因为一次模型抽风而整体挂掉。
        print(f"        [警告] 裁判 JSON 解析失败：{exc}")
        return {}


def _harmonic(precision: float, recall: float) -> float:
    """F1 = 2PR/(P+R)；P+R 为 0 时约定返回 0，避免除零。

    这个「约定的 0」很重要：如果返回 None 或者抛异常，
    ToolCallF1 在下游就没法参与均值汇总了。
    """
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

### 3.4 四个指标函数

指标一 · 主题一致性（TopicAdherence）——LLM 判定「答了哪些允许主题 / 拒了哪些 / 越界聊了什么」，
再套 TP/FP/FN → P/R/F1。**注意它判的是「过程」，不是「结果」。**

In [ ]:
# ---------- 5. 指标一：主题一致性 TopicAdherence ----------
# 公式与判定口径全部来自课案：「LLM 判定答了哪些 / 拒了哪些 / 越界聊了什么」，
# 再套 TP/FP/FN → precision/recall/F1。**注意它判的是「过程」，不是「结果」。**
def _canonical_topic(name: str, whitelist: list[str]) -> str | None:
    """把裁判模型说的主题名对回白名单里的标准写法。

    为什么要这一步：LLM 常把「数学计算」说成「数学运算」、把「Python」说成「Python版本」。
    直接拿字符串做 in 判断会把这类情况误判成「越界」，指标就假性掉分。
    这里去掉空白与标点后做双向包含匹配，口径宽松但结果稳定。
    """
    def squeeze(text: str) -> str:
        # 去掉空白与中英文标点再比：这样「数学-计算」「数学计算」会被视为同一个主题
        return re.sub(r"[\s\-_/·、，,。.（）()]", "", str(text)).lower()

    key = squeeze(name)
    if not key:
        return None
    for topic in whitelist:
        topic_key = squeeze(topic)
        # 双向包含：容忍模型多写（「Python版本」⊃「Python」）和少写（「Python」⊂「Python版本」）
        if topic_key and (topic_key in key or key in topic_key):
            return topic          # 返回**白名单里的原词**，保证统计口径统一
    return None


def topic_adherence(messages, reference_topics: list[str], mode: str = "f1") -> dict:
    """主题一致性：LLM 判定「答了哪些允许主题 / 拒了哪些 / 越界聊了什么」，再套公式。

    TP = 已回答且符合参考主题
    FP = 已回答但不在参考主题里（越界）
    FN = 参考主题里被拒绝回答的（漏答）

    返回 {"score": 分数, "precision": ..., "recall": ..., "f1": ..., "detail": {...}}
    mode 支持 "precision" / "recall" / "f1"（默认，与 ragas 的 TopicAdherence 一致）。
    """
    trajectory = format_trajectory(messages)
    data = judge(
        f"允许 Agent 回答的主题白名单：{json.dumps(reference_topics, ensure_ascii=False)}\n\n"
        f"Agent 的完整对话轨迹：\n{trajectory}\n\n"
        "请判断下面三件事：\n"
        "1) 白名单里哪些主题 Agent 确实正面回答/处理了（照抄白名单里的原词）；\n"
        "2) 白名单里哪些主题 Agent 拒绝、推脱或回避了（照抄白名单里的原词）；\n"
        "3) Agent 是否还聊了白名单之外的主题？有就列出来，没有就给空数组。\n"
        '只输出 JSON：{"answered_topics": [], "refused_topics": [], "off_topic_subjects": []}'
    )

    # 1、2 两项只在白名单里取值：先做同义归一，再统计
        # TP/FP/FN 的语义按课案定义：答了允许的主题是 TP，越界是 FP，该答却拒了是 FN。
    answered, refused, unmapped = [], [], []
    for name in dict.fromkeys(data.get("answered_topics", [])):
        topic = _canonical_topic(name, reference_topics)
        (answered if topic else unmapped).append(topic or name)
    for name in dict.fromkeys(data.get("refused_topics", [])):
        topic = _canonical_topic(name, reference_topics)
        if topic:
            refused.append(topic)
    # 裁判在第一项里给了白名单之外的主题 → 按「回答过但不被允许」计入 FP
            # 只统计白名单里的主题：裁判多说的主题归入 unmapped，下一步按 FP 处理。
    off_topic = list(dict.fromkeys(list(data.get("off_topic_subjects", [])) + unmapped))

    tp, fp, fn = len(answered), len(off_topic), len(refused)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = _harmonic(precision, recall)
# mode 决定返回哪个分数：默认 f1，与 ragas 的 TopicAdherence 一致。
    score = {"precision": precision, "recall": recall, "f1": f1}.get(mode, f1)

    return {
        "score": round(score, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "detail": {"TP(回答了允许的主题)": answered, "FP(越界聊的主题)": off_topic,
                   "FN(该答却拒绝的主题)": refused},
    }

指标二/三 · 工具调用准确率（ToolCallAccuracy，二元）与 F1（ToolCallF1，给部分分）：

In [ ]:
# ---------- 6. 指标二：工具调用准确率 ToolCallAccuracy ----------
def _norm_args(args: dict) -> dict:
    """参数归一化：把字符串参数里的空白全部去掉。

    只用于本文件额外演示的「宽松比对」——真实模型常把
    '15 * 8 + 23' 写成 '15*8+23'，语义一样但字符串不等，
    课案的默认口径是**精确匹配**，所以默认不启用归一化。
    """
    return {k: (re.sub(r"\s+", "", v) if isinstance(v, str) else v) for k, v in args.items()}


def _call_key(tc, normalize: bool = False) -> tuple:
    args = _norm_args(tc.args) if normalize else tc.args
    return tc.name, json.dumps(args, sort_keys=True, ensure_ascii=False)


def tool_call_accuracy(actual_calls: list, reference_calls: list,
                       strict_order: bool = True, normalize: bool = False) -> float:
    """工具调用准确率：二元评分，完全匹配得 1，否则 0。

    strict_order=True（课案默认）要求顺序也一致；实际调用数不等直接判 0。
    """
    if len(actual_calls) != len(reference_calls):
        return 0.0
    if not reference_calls:
        return 1.0                          # 都不需要调工具，也算完全匹配

    a_keys = [_call_key(tc, normalize) for tc in actual_calls]
    r_keys = [_call_key(tc, normalize) for tc in reference_calls]
    if strict_order:
        return 1.0 if a_keys == r_keys else 0.0
    # 非严格顺序：只要求集合（含重复次数）一致
    return 1.0 if Counter(a_keys) == Counter(r_keys) else 0.0


# ---------- 7. 指标三：工具调用 F1 分数 ToolCallF1 ----------
def tool_call_confusion(actual_calls: list, reference_calls: list,
                        normalize: bool = False) -> dict:
    """按「无序多重集匹配」算出混淆矩阵 —— 这是 F1 的计算基础。

    匹配规则和课案一致：工具**名称与参数完全相同**才算一对。
    因为是按等价关系配对（相等就一定能配），
    最大匹配数 = 各调用键在两个多重集里的交集大小：

        TP = Σ_key min(实际里该 key 的个数, 期望里该 key 的个数)
        FP = 实际调用总数 - TP   （多调了的：非预期的额外调用）
        FN = 期望调用总数 - TP   （漏调了的：预期但没进行的调用）

    用多重集而不是集合，是为了正确处理「同一个工具被调用两次」的情况。
    """
        # 用多重集而不是集合，是为了正确处理「同一个工具被调用两次」的情况。
    actual_keys = Counter(_call_key(tc, normalize) for tc in actual_calls)
    expected_keys = Counter(_call_key(tc, normalize) for tc in reference_calls)
    tp = sum(min(actual_keys[k], expected_keys[k]) for k in actual_keys.keys() | expected_keys.keys())

    def _brief(counter: Counter) -> list[str]:
        """把「多调 / 漏调」压成 ['web_search×4'] 这种形式，避免刷屏。"""
        return [f"{name}×{n}" if n > 1 else name for (name, _), n in counter.items()]

    return {
        "TP": tp,
        "FP": len(actual_calls) - tp,
        "FN": len(reference_calls) - tp,
        "多调的调用": _brief(actual_keys - expected_keys),
        "漏调的调用": _brief(expected_keys - actual_keys),
    }


def tool_call_f1(actual_calls: list, reference_calls: list, normalize: bool = False) -> dict:
    """工具调用 F1：精确率、召回率、F1（无序匹配，给部分分）。"""
    cm = tool_call_confusion(actual_calls, reference_calls, normalize)
    tp, fp, fn = cm["TP"], cm["FP"], cm["FN"]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    # P+R 为 0 时 _harmonic 约定返回 0，避免除零把整条评估打断。
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return {
        "score": round(_harmonic(precision, recall), 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "confusion": cm,
    }

指标四 · 智能体目标准确率（AgentGoalAccuracy）——LLM 裁判判定「用户的核心目标有没有被真正实现」。
有 `reference` 走 `WithReference` 模式，没有走 `WithoutReference`（让 LLM 自己推断目标）。

In [ ]:
# ---------- 8. 指标四：智能体目标准确率 AgentGoalAccuracy ----------
def agent_goal_accuracy(messages, reference: str | None = None) -> dict:
    """智能体目标准确率：LLM 裁判判定「用户的核心目标有没有被真正实现」，1 或 0。

    reference 给不给，对应课案说的两种模式：
        给了     → AgentGoalAccuracyWithReference（拿参考目标做对比，更客观）
        没给     → AgentGoalAccuracyWithoutReference（让 LLM 自己从对话里推断目标）
    """
    trajectory = format_trajectory(messages)
            # reference 给不给，对应课案的两种模式：WithReference 更客观，WithoutReference 更通用。
    if reference:
        prompt = (
            f"用户的原始请求：{messages[0].content}\n\n"
            f"参考目标（期望 Agent 达成的结果）：{reference}\n\n"
            f"Agent 的完整轨迹与最终回答：\n{trajectory}\n\n"
            "请判断 Agent 是否真正实现了用户的目标 —— 要看结果对不对（算得准不准、"
            "信息有没有给到、结论有没有下），不要只看态度好不好。\n"
            '只输出 JSON：{"achieved": true 或 false, "reason": "一句话理由"}'
        )
        mode = "AgentGoalAccuracyWithReference"
    else:
        prompt = (
            f"Agent 的完整对话轨迹：\n{trajectory}\n\n"
            "请先从对话里推断出用户真正想达成的目标，再判断 Agent 是否实现了它。\n"
            '只输出 JSON：{"goal": "你推断出的用户目标", "achieved": true 或 false, "reason": "一句话理由"}'
        )
        mode = "AgentGoalAccuracyWithoutReference"

    # 让裁判「看结果对不对」，而不是看态度好不好 —— 否则礼貌的空话也会被判成达成目标。
    data = judge(prompt)
    achieved = bool(data.get("achieved"))
    return {
        "score": 1.0 if achieved else 0.0,
        "mode": mode,
        "reason": data.get("reason", "(裁判未给出理由)"),
        "inferred_goal": data.get("goal"),
    }

### 3.5 评估数据集与主循环

三条用例是刻意配出来的「三种典型失败模式」，跑完对照着看才懂四个指标为什么要分开：

| 用例 | 故意设的坑 | 预期现象 |
|---|---|---|
| `15*8+23` | 参考参数带空格（`15 * 8 + 23`），模型实际回无空格 | Accuracy 0、F1 0，宽松匹配满分 |
| 搜 Python 版本 | 模型多调一次 `web_search`（换语言重搜） | Accuracy 0、F1 0.67 |
| 天气 + 温度换算 | 多调 1 次 + 漏调 1 次 | Accuracy 0、F1 0.50 |

In [ ]:
# ---------- 9. 评估数据集（课案原文三条）与主流程 ----------
DATASET_NAME = "agent_evaluation_v2"

DATASET_ITEMS = [
    {
        "input": "帮我计算 15 * 8 + 23",
        "expected_output": "143",
        "metadata": {
            # reference_topics 是主题白名单；模型答了「数学计算」以外的话题就算越界(FP)
            "reference_topics": ["数学计算"],
            # 注意这里参数带空格：模型实际会回 "15*8+23"，于是精确匹配必然失败
            "reference_tool_calls": [{"name": "calculator", "args": {"expression": "15 * 8 + 23"}}],
        },
    },
    {
        "input": "搜索一下 Python 最新版本是什么",
        "expected_output": "Python 最新稳定版本是 3.13，2024 年 10 月发布",
        "metadata": {
            "reference_topics": ["Python", "编程语言版本"],
            "reference_tool_calls": [{"name": "web_search", "args": {"query": "Python 最新版本"}}],
        },
    },
    {
        "input": "先查一下北京天气，再算 30°C 转华氏度（公式：F = C * 9/5 + 32）",
        "expected_output": "北京今天晴，30°C；华氏度为 86°F",
        "metadata": {
            "reference_topics": ["天气查询", "温度转换"],
            # 这条是**多步任务**：顺序 get_weather → calculator 有语义，所以 Accuracy 用 strict_order=True
            "reference_tool_calls": [
                {"name": "get_weather", "args": {"city": "北京"}},
                {"name": "calculator", "args": {"expression": "30 * 9/5 + 32"}},
            ],
        },
    },
]

In [ ]:
# ---------- 10. 单条用例的完整评估流程（四个指标一起算） ----------
def evaluate_one(item) -> dict:
    """跑一条测试项，算四个指标，返回结果字典。

    顺序很重要：先拿到**真实轨迹**，再基于轨迹算四个指标。
    四个指标吃的是同一份轨迹，所以它们之间不会互相影响，可以横向比较。
    """
    # 把数据集里的 metadata 还原成评估用的对象（字段名与 ragas 一致）
    reference_topics = item.metadata.get("reference_topics", [])
    reference_tool_calls = [ToolCall(name=tc["name"], args=tc.get("args", {}))
                            for tc in item.metadata.get("reference_tool_calls", [])]

    # 1) 真跑 Agent，拿到真实轨迹（带步数上限保护）
    messages, truncated = run_agent(item.input)
    # 从后往前找第一条有内容的 AI 消息 —— 最后一条消息不一定是最终回答
    # （DeepAgents 收尾时可能追加一条空 content 的工具调用消息）
    answer = ""
    for msg in reversed(messages):
        if isinstance(msg, LCAI) and msg.content:
            answer = str(msg.content)
            break
    trajectory = to_eval_messages(messages)          # 过滤掉内置工具，只留被测行为
    actual_calls = collect_actual_tool_calls(trajectory)

    # 2) 四个指标（全部本地可算，不依赖 Langfuse）
    topic = topic_adherence(trajectory, reference_topics, mode="f1")
    accuracy = tool_call_accuracy(actual_calls, reference_tool_calls, strict_order=True)
    f1 = tool_call_f1(actual_calls, reference_tool_calls)
    # 额外观察：同一批调用换成「忽略空白的宽松匹配」后 F1 会变成多少
    # 这一列是本文件最有教学价值的设计 —— 它把「Accuracy=0 到底是模型差还是口径严」当场问清楚
    f1_loose = tool_call_f1(actual_calls, reference_tool_calls, normalize=True)
    # 目标达成用 expected_output 当 reference，即课案的 AgentGoalAccuracyWithReference 模式
    goal = agent_goal_accuracy(trajectory, reference=str(item.expected_output))

    return {
        "item": item, "answer": answer, "trajectory": trajectory, "truncated": truncated,
        "actual_calls": actual_calls, "reference_calls": reference_tool_calls,
        "topic": topic, "accuracy": accuracy, "f1": f1, "f1_loose": f1_loose, "goal": goal,
    }

In [ ]:
# ---------- 11. 主循环：逐条评估 → 打分上报 → 四指标汇总 ----------
def main_loop():
    print("\n" + "=" * 72)
    print(f"数据集 {DATASET_NAME}：逐条跑 Agent → 四个指标打分 → 上报")
    print("=" * 72)

    if LANGFUSE_READY:
        # 课案流程：先 create_dataset 建空集（已存在则返回已有的），再逐条 add，最后 get 回来
        langfuse.create_dataset(name=DATASET_NAME)
        for item in DATASET_ITEMS:
            langfuse.create_dataset_item(dataset_name=DATASET_NAME, **item)
        items = langfuse.get_dataset(DATASET_NAME).items
    else:
        print(f"[降级] 本应创建 Langfuse 数据集 {DATASET_NAME}（{len(DATASET_ITEMS)} 条测试项）")
        # SimpleNamespace 的字段名（input / expected_output / metadata）与 dataset.items 一致，
        # 于是下面整个 for 循环一行都不用改 —— 降级路径不产生「另一套代码」
        items = [SimpleNamespace(**item) for item in DATASET_ITEMS]

    summary = []
    for idx, item in enumerate(items, start=1):
        print(f"\n[{idx}/{len(items)}] Q: {item.input}")
        r = evaluate_one(item)

        # —— 先把「事实」打出来：实际调了什么、参考期望什么、有没有被截断 ——
        print(f"        实际工具调用：{format_calls(r['actual_calls'])}")
        print(f"        参考工具调用：{format_calls(r['reference_calls'])}")
        if r["truncated"]:
            print(f"        ⚠ 本次执行触到步数上限 {AGENT_STEP_LIMIT}，指标按已完成的轨迹计算")
        print(f"        A: {r['answer'][:70] or '(无最终回答)'}…")

        # —— 再逐项打指标：每个指标都同时给出分数和「为什么是这个分数」 ——
        # ① 主题一致性：看的是「过程有没有跑题」，所以给 P/R 和 TP/FP/FN 明细
        print(f"        ① 主题一致性 F1 = {r['topic']['score']:.2f}"
              f"  (P={r['topic']['precision']:.2f} R={r['topic']['recall']:.2f})")
        print(f"           明细：{json.dumps(r['topic']['detail'], ensure_ascii=False)}")
        # ② 工具调用准确率：二元，只要名称/参数/顺序有一处不一致就是 0
        print(f"        ② 工具调用准确率 = {r['accuracy']:.2f}（二元，名称+参数+顺序完全匹配）")
        # ③ 工具调用 F1：给部分分，所以必须把混淆矩阵打出来，否则 0.5 这种分数看不懂
        cm = r["f1"]["confusion"]
        print(f"        ③ 工具调用 F1 = {r['f1']['score']:.2f}"
              f"  (P={r['f1']['precision']:.2f} R={r['f1']['recall']:.2f})")
        print(f"           混淆矩阵：TP={cm['TP']} FP={cm['FP']} FN={cm['FN']}"
              f"  多调={cm['多调的调用']} 漏调={cm['漏调的调用']}")
        # 只在宽松匹配算出不同分数时才打印，避免每条都刷一句废话
        if r["f1_loose"]["score"] != r["f1"]["score"]:
            print(f"           （若改成忽略空白的宽松匹配：F1 = {r['f1_loose']['score']:.2f}"
                  f" —— 精确匹配太严，这就是 F1 存在的意义）")
        # ④ 智能体目标准确率：0/1 二值，所以用 :.0f；裁判理由一定要打，否则学员不知道它为什么这么判
        print(f"        ④ 智能体目标准确率 = {r['goal']['score']:.0f}  [{r['goal']['mode']}]")
        print(f"           裁判理由：{r['goal']['reason']}")
        if r["goal"]["inferred_goal"]:
            print(f"           推断的用户目标：{r['goal']['inferred_goal']}")

        # —— 最后上报：四个指标都挂在同一条 trace 上，name 就是指标名 ——
        # 真实环境里 tid 应该来自 langfuse_handler.last_trace_id；
        # 这里用一个可预测的假 id，方便对照「本应上报的报文」
                     # 四个分数都挂在同一条 trace 上，看板才能按指标名聚合、按时间对比。
        tid = "trace-agent-%03d" % idx
        report_score(trace_id=tid, name="Topic Adherence", value=r["topic"]["score"],
                     comment=f"参考主题: {r['item'].metadata.get('reference_topics')}")
        report_score(trace_id=tid, name="Tool Call Accuracy", value=r["accuracy"],
                     comment=f"期望工具: {[tc.name for tc in r['reference_calls']]}")
        report_score(trace_id=tid, name="Tool Call F1", value=r["f1"]["score"], comment="调和平均")
        report_score(trace_id=tid, name="Agent Goal Accuracy", value=r["goal"]["score"],
                     comment=f"期望: {str(r['item'].expected_output)[:40]}…")
        print("-" * 50)

        summary.append((item.input, r))

    if LANGFUSE_READY:
        langfuse.flush()          # SDK 默认异步批量上报，不 flush 分数可能还没到服务端

    # ---------- 汇总：四个指标的均值，就是一次实验的总体结果 ----------
    print("\n" + "=" * 72)
    print("四指标汇总（各测试项均值）")
    print("=" * 72)
    n = len(summary)
    # 每行都附一句「这个数该怎么读」，否则四个 0~1 的数字看不出门道
    # 每行都附一句「这个数该怎么读」，否则四个 0~1 的数字看不出门道。
    print(f"  ① 主题一致性 Topic Adherence      : {sum(r['topic']['score'] for _, r in summary) / n:.3f}")
    print(f"  ② 工具调用准确率 Tool Call Accuracy: {sum(r['accuracy'] for _, r in summary) / n:.3f}"
          f"   ← 二元打分，只要有一处参数写法不同就掉到 0")
    print(f"  ③ 工具调用 F1 Tool Call F1         : {sum(r['f1']['score'] for _, r in summary) / n:.3f}"
          f"   ← 同为 0 分的项，这里能看出「接近正确」的程度")
    print(f"  ④ 智能体目标准确率 Agent Goal Acc.  : {sum(r['goal']['score'] for _, r in summary) / n:.3f}")
    print("\n  Accuracy 是全有全无，F1 给部分分：迭代早期看 F1 找方向，验收阶段看 Accuracy 卡线。")

In [ ]:
# ---------- 12. 入口：先讲清「本机缺什么」，再真跑 ----------
print("=" * 72)
print("Agent 评估四指标：主题一致性 / 工具调用准确率 / 工具调用 F1 / 智能体目标准确率")
print("=" * 72)
# 把「消息结构体来自 ragas 还是本地等价实现」显式打出来：
# 这决定了分数的口径，学员看到数字时必须知道是哪种
print(f"评估轨迹的消息结构体来自：{MESSAGE_SOURCE}")
print("本文件的四个评估函数都是纯数据进、纯数据出，不依赖 Langfuse 与 ragas，")
print("所以下面会真跑 Agent、真调 LLM 当裁判，把每个指标的数值算出来。")

# 缺 Langfuse 只影响「上报」，不影响「算分」—— 这段说明就是为了让学员别误以为跑不了
if not LANGFUSE_READY:
    print("\n【进入降级演示】Langfuse 密钥为空")
    print("  settings.langfuse_public_key = '' ，settings.langfuse_secret_key = ''")
    print("-" * 72)
    print("四个指标的数值照常真算；只是「上报分数」这一步改成打印报文。")
    print("要看到真实看板，按课案「安装」一节准备环境：")
    # 缺 Langfuse 只影响上报：这段说明就是为了让学员别误以为「跑不了」。
    print("  1) git clone https://github.com/langfuse/langfuse.git")
    print("     cd langfuse")
    print("     docker compose up -d          # 启动后访问 http://localhost:3000")
    print("  2) 首次注册的账号即为管理员；新建项目 → Settings → API Keys → 创建密钥")
    print("  3) 写入 F:\\ProGram\\Python_Base\\.env ：")
    print("        LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("        LANGFUSE_SECRET_KEY=sk-lf-...")
    print("        LANGFUSE_HOST=http://localhost:3000    # 本地 docker 部署用这个")
    print("  4) 重新运行本脚本，四项分数会挂到每条 trace 上")
    print("=" * 72)

main_loop()

### 预期输出

```text
========================================================================
Agent 评估四指标：主题一致性 / 工具调用准确率 / 工具调用 F1 / 智能体目标准确率
========================================================================
评估轨迹的消息结构体来自：本地等价结构体（未安装 ragas）
本文件的四个评估函数都是纯数据进、纯数据出，不依赖 Langfuse 与 ragas，
所以下面会真跑 Agent、真调 LLM 当裁判，把每个指标的数值算出来。
========================================================================
数据集 agent_evaluation_v2：逐条跑 Agent → 四个指标打分 → 上报
========================================================================
[1/N] Q: 帮我计算 15 * 8 + 23
        实际工具调用：...
        参考工具调用：...
        ① 主题一致性 F1 = ...  (P=... R=...)
        ② 工具调用准确率 = ...（二元，名称+参数+顺序完全匹配）
        ③ 工具调用 F1 = ...  (P=... R=...)
        ④ 智能体目标准确率 = ...  [AgentGoalAccuracyWithReference]
...
========================================================================
四指标汇总（各测试项均值）
========================================================================
  ① 主题一致性 Topic Adherence      : ...
  ② 工具调用准确率 Tool Call Accuracy: ...
  ③ 工具调用 F1 Tool Call F1         : ...
  ④ 智能体目标准确率 Agent Goal Acc.  : ...
```

> ⚠️ **本格所有分数与裁判理由都由模型决定，是本课最不稳定的一格，别逐字比对。**
> 上面 `N` 是 `agent_evaluation_v2` 数据集累积的条目数（追加式，重跑一次多 3 条）。
> 源文件头里记的本机实测均值（3 条用例）是：主题一致性 1.000 / 工具调用准确率 **0.000** /
> 工具调用 F1 0.389 / 智能体目标准确率 1.000 —— 这三条结论才是本节的真正考点：
> ① Accuracy 全 0 是「精确匹配」口径太硬（参数只差空格）；② F1 能区分「错得有多离谱」；
> ③ 结果对了（Goal=1）≠ 过程规范（Accuracy=0）。

## 小结

- **打分**就是把「主观好坏」变成可筛选字段：`trace_id` 定位调用、`name` 决定聚合、`comment` 留下依据；
- **类别型分数必须显式写 `data_type="CATEGORICAL"`**，否则被当 NUMERIC 解析字符串而报错；
- **精确字符串匹配只能当玩具**：模型换个说法（哪怕答对）就拿 0 分，语义正确必须上 LLM 裁判；
- **RAG 评估**分两组：检索看 CP / CR（精与全），生成看 Faithfulness / AR（幻觉与切题）；
- **Agent 评估**看的是「行为」不是「回答」：Accuracy 是全有全无，F1 给部分分——
  迭代早期看 F1 找方向，验收阶段看 Accuracy 卡线；
- 结果对了（Goal=1）不等于过程规范（Accuracy=0），验收时两个指标要一起看。

## 常见坑

1. **本机 Langfuse 是 v4 的 events_only 模式**：`GET /api/public/v2/scores` 这类**取分**端点
   不存在（`404 This endpoint is not available on deployments running in Langfuse v4
   events_only mode.`）。但**打分上传**（`create_score` / `score_current_trace`）走事件通道、
   照常可用；UI 上也能照常看到分数。程序取数要走 v4 的 metrics 接口。
2. **类别型分数（`data_type="CATEGORICAL"`）的值是字符串**（如 `"thumb_up"`）——这是
   `create_score` 的正确用法；别和「标注队列的 ScoreConfig」搞混：那儿的
   `categories[].value` **必须是数字**（`label` 才是给人看的显示名），写字符串会被服务端拒。
3. **`score_current_trace` 不需要 `trace_id`**：只在 `@observe`（或 `start_as_current_*`）
   上下文里调用才有效，因为它从当前 OTEL span 里取 trace id；脱离上下文调用是静默 no-op。
4. **`flush()` 是异步批量上报的开关**：不 `flush()`，分数可能还没到服务端脚本就退出了。
5. **评测模型最好和业务模型分开**（课案用 DashScope 就是这个原因），「自己评自己」容易偏袒；
   而且 AnswerRelevancy 需要句向量，通用 chat 模型通常没有 embedding 接口。
6. **评估脚本必须扛住模型自由发挥**：裁判要求只回 JSON，模型却爱包 ```json``` 或用错类型——
   用正则抠 `{...}` + 解析失败兜底 0 分，绝不让一条评估项把整轮评估带崩。
7. **Agent 评估要设 `recursion_limit`**：模型偶尔陷入「反复调同一工具」的循环，
   不设上限一条用例能跑几十分钟、刷几万行。
8. **数据集是追加式的，重跑会累积条目**：`create_dataset_item` 每次都是**新增**，
   `get_dataset(name).items` 返回的是**当前全部**条目 —— 所以密钥配置好之后，
   第 1.2 / 2.2 / 3 节的评估循环会越跑条目越多、越跑越久（本机这次已是 18 条）。
   想回到干净的 3 条，去 Langfuse 的 Datasets 页删掉同名数据集重建，或改用带时间戳的数据集名。

## 官方链接

- 评估总览：<https://langfuse.com/docs/evaluation/overview>
- 打分（Scores）：<https://langfuse.com/docs/evaluation/features/scores>
- 实验（Experiments）：<https://langfuse.com/docs/evaluation/experiments/experiments>
- 数据集（Datasets）：<https://langfuse.com/docs/evaluation/experiments/datasets>
- RAG 评估（Ragas）：<https://langfuse.com/docs/evaluation/ragas-evaluation>